# Tech Challenge — Fase 1

**Tema:** Saúde da mulher — classificação de câncer de mama (maligno vs benigno)

**Objetivo:** atender às entregas técnicas do PDF — EDA, pré-processamento, dois ou mais modelos, métricas e explicabilidade.

## 1. Contexto e definição do problema

Redes de saúde feminina precisam **triagem rápida** para priorizar casos de risco. Este trabalho não substitui o médico: é um **apoio à decisão** baseado em padrões históricos dos dados.

- **Entrada:** medidas numéricas de núcleos celulares (ex.: raio, textura, perímetro).
- **Saída:** classe `malignant` (1) ou `benign` (0).
- **Métrica prioritária sugerida:** **recall** da classe maligna — errar um caso maligno (falso negativo) costuma ser mais grave que um falso positivo.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

## Dicionário de atributos

- **id**: Identificador único da amostra.
- **diagnosis**: Diagnóstico original no dataset (`M` = maligno, `B` = benigno).
- **radius_mean**: Raio médio dos núcleos celulares.
- **texture_mean**: Textura média dos núcleos celulares.
- **perimeter_mean**: Perímetro médio dos núcleos celulares.
- **area_mean**: Área média dos núcleos celulares.
- **smoothness_mean**: Suavidade média da borda dos núcleos.
- **compactness_mean**: Compacidade média dos núcleos.
- **concavity_mean**: Concavidade média das formas dos núcleos.
- **concave points_mean**: Número médio de pontos côncavos nos núcleos.
- **symmetry_mean**: Simetria média dos núcleos.
- **fractal_dimension_mean**: Dimensão fractal média dos núcleos.
- **radius_se**: Erro padrão do raio dos núcleos.
- **texture_se**: Erro padrão da textura dos núcleos.
- **perimeter_se**: Erro padrão do perímetro dos núcleos.
- **area_se**: Erro padrão da área dos núcleos.
- **smoothness_se**: Erro padrão da suavidade dos núcleos.
- **compactness_se**: Erro padrão da compacidade dos núcleos.
- **concavity_se**: Erro padrão da concavidade dos núcleos.
- **concave points_se**: Erro padrão do número de pontos côncavos.
- **symmetry_se**: Erro padrão da simetria dos núcleos.
- **fractal_dimension_se**: Erro padrão da dimensão fractal.
- **radius_worst**: Maior valor observado de raio nos núcleos.
- **texture_worst**: Maior valor observado de textura nos núcleos.
- **perimeter_worst**: Maior valor observado de perímetro nos núcleos.
- **area_worst**: Maior valor observado de área nos núcleos.
- **smoothness_worst**: Maior valor observado de suavidade nos núcleos.
- **compactness_worst**: Maior valor observado de compacidade nos núcleos.
- **concavity_worst**: Maior valor observado de concavidade nos núcleos.
- **concave points_worst**: Maior valor observado de pontos côncavos nos núcleos.
- **symmetry_worst**: Maior valor observado de simetria nos núcleos.
- **fractal_dimension_worst**: Maior dimensão fractal observada nos núcleos.

Observação: a coluna Unnamed: 32 pode ser descartada, pois não contém valores relevantes.

## 2. Carregamento e exploração dos dados

In [ ]:
DATA_PATH = Path("..") / "data" / "data.csv"

df = pd.read_csv(DATA_PATH)

# Remover colunas vazias e linhas/colunas com valores nulos
df = df.dropna(axis=1, how='all')
df = df.dropna()

# Alvo: M = maligno (1), B = benigno (0)
df["target"] = (df["diagnosis"].astype(str).str.upper() == "M").astype(int)

# Removendo colunas que não serão utilizadas
df = df.drop(columns=["id", "diagnosis"])
feature_cols = [c for c in df.columns if c != "target"]

print(f"Linhas: {df.shape[0]} | Colunas: {len(feature_cols)}")
print(df["target"].value_counts())
df.head()

#

In [33]:
df.info()
df.describe().T.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             569 non-null

,count,mean,std,min,25%,50%,75%,max
id,569.0,3.037183e+07,1.250206e+08,8670.00000,869218.00000,906024.00000,8.813129e+06,9.113205e+08
radius_mean,569.0,1.412729e+01,3.524049e+00,6.98100,11.70000,13.37000,1.578000e+01,2.811000e+01
texture_mean,569.0,1.928965e+01,4.301036e+00,9.71000,16.17000,18.84000,2.180000e+01,3.928000e+01
perimeter_mean,569.0,9.196903e+01,2.429898e+01,43.79000,75.17000,86.24000,1.041000e+02,1.885000e+02
area_mean,569.0,6.548891e+02,3.519141e+02,143.50000,420.30000,551.10000,7.827000e+02,2.501000e+03
smoothness_mean,569.0,9.636028e-02,1.406413e-02,0.05263,0.08637,0.09587,1.053000e-01,1.634000e-01
compactness_mean,569.0,1.043410e-01,5.281276e-02,0.01938,0.06492,0.09263,1.304000e-01,3.454000e-01
concavity_mean,569.0,8.879932e-02,7.971981e-02,0.00000,0.02956,0.06154,1.307000e-01,4.268000e-01
concave points_mean,569.0,4.891915e-02,3.880284e-02,0.00000,0.02031,0.03350,7.400000e-02,2.012000e-01
symmetry_mean,569.0,1.811619e-01,2.741428e-02,0.10600,0.16190,0.17920,1.957000e-01,3.040000e-01


## 3. Análise de correlação

In [ ]:
corr = df.corr(numeric_only=True)

# Correlação de cada feature com o target, ordenada
corr_target = corr["target"].drop("target").sort_values(ascending=False)
print("Top 10 features mais correlacionadas com o target:")
print(corr_target.head(10))
print("\nTop 10 features menos correlacionadas com o target:")
print(corr_target.tail(10))

# Heatmap geral de correlação
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Mapa de correlação entre features e target")
plt.show()

In [ ]:
# Pares de features com alta multicolinearidade (|corr| > 0.9, excluindo target)
feat_corr = corr.drop(index="target", columns="target")
pairs = (
    feat_corr.where(np.triu(np.ones(feat_corr.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
high_corr_pairs = pairs[pairs.abs() > 0.9]
print(f"{len(high_corr_pairs)} pares de features com |correlação| > 0.9:")
high_corr_pairs

## 4. Pré-processamento e separação treino/teste

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Features removidas por baixa correlação com target (|corr| < 0.1)
low_corr = ["smoothness_se", "fractal_dimension_mean", "texture_se",
            "symmetry_se", "fractal_dimension_se"]

# Features removidas por multicolinearidade — mantendo a de maior corr com target em cada grupo
multicolinear = [
    "radius_mean", "area_mean",         # grupo tamanho _mean → fica perimeter_mean
    "radius_worst", "area_worst",       # grupo tamanho _worst → fica perimeter_worst
    "perimeter_se", "area_se",          # grupo tamanho _se → fica radius_se
    "concavity_mean", "concave points_mean",  # grupo concavidade → fica concave points_worst
    "texture_mean",                     # grupo textura → fica texture_worst
]

drop_cols = low_corr + multicolinear

X = df.drop(columns=drop_cols + ["target"])
y = df["target"]

print(f"Features removidas: {len(drop_cols)}")
print(f"Features mantidas: {X.shape[1]}")
print(list(X.columns))

In [ ]:
# Split treino/teste — 80/20, estratificado para manter proporção de classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")
print(f"Distribuição treino — benigno: {(y_train==0).sum()} | maligno: {(y_train==1).sum()}")
print(f"Distribuição teste  — benigno: {(y_test==0).sum()}  | maligno: {(y_test==1).sum()}")

In [ ]:
# Padronização: fit apenas no treino, aplica no teste para evitar data leakage
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Verificação: após escalar, média ≈ 0 e desvio ≈ 1 no treino
means = X_train_sc.mean(axis=0).round(6)
stds  = X_train_sc.std(axis=0).round(3)
print(f"Média das features no treino (deve ser ≈ 0): {means[:5]}")
print(f"Desvio das features no treino (deve ser ≈ 1): {stds[:5]}")

## 5. Modelagem — três algoritmos

## 6. Explicabilidade — importância das variáveis e SHAP

## 7. Discussão crítica